In [2]:
!pip install PyPDF2 scikit-learn

In [3]:
!pip install nltk

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached click-8.3.3-py3-none-any.whl (110 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nltk]3/4 [nltk]


In [4]:
import os
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

In [5]:
def extrair_texto_pdf(caminho):
    texto = ""
    try:
        reader = PdfReader(caminho)
        for page in reader.pages:
            texto += page.extract_text() or ""
    except:
        pass
    return texto

In [6]:
texts = []
labels = []

base_path = "/Users/isacosta/Documents/M. Defesa/Representacoes-ML"

for label in ["representacao", "nao_representacao"]:
    pasta = os.path.join(base_path, label)
    
    for arquivo in os.listdir(pasta):
        if arquivo.endswith(".pdf"):
            caminho = os.path.join(pasta, arquivo)
            texto = extrair_texto_pdf(caminho)
            
            texts.append(texto)
            labels.append(label)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

In [8]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isacosta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from nltk.corpus import stopwords

stopwords_pt = stopwords.words('portuguese')

vectorizer = TfidfVectorizer(stop_words=stopwords_pt)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

                   precision    recall  f1-score   support

nao_representacao       1.00      1.00      1.00        11
    representacao       1.00      1.00      1.00        10

         accuracy                           1.00        21
        macro avg       1.00      1.00      1.00        21
     weighted avg       1.00      1.00      1.00        21



In [11]:
import json
import numpy as np

# salvar vocabulario
vocab = vectorizer.vocabulary_
idf = vectorizer.idf_.tolist()

# salvar modelo NB
model_data = {
    "class_log_prior": model.class_log_prior_.tolist(),
    "feature_log_prob": model.feature_log_prob_.tolist(),
    "classes": model.classes_.tolist()
}

# salvar tudo
with open("model.json", "w") as f:
    json.dump({
        "vocab": vocab,
        "idf": idf,
        "model": model_data,
        "stopwords": stopwords_pt
    }, f)